Flores do Brasil: 

**Fonte:** [GBIF — Global Biodiversity Information Facility](https://www.gbif.org/)  
**Escopo:** Registros de ocorrência de plantas com flor nativas do Brasil, com coordenadas geográficas validadas.

Pipeline com quatro etapas em sequência:

1. **Coleta** — requisições paginadas à API REST do GBIF  
2. **Limpeza** — seleção de colunas, remoção de nulos, validação de coordenadas, deduplicação  
3. **Armazenamento** — exportação para `.parquet` e `.csv`  
4. **Resumo** — estatísticas descritivas do dataset final

In [ ]:
import requests
import pandas as pd
import time
import os

### 1. Configuração

Parâmetros da busca na API e quantidade de registros alvo.  
O GBIF retorna no máximo 300 itens por página e o pipeline itera com offset até atingir o total desejado.

In [16]:
BASE_URL = "https://api.gbif.org/v1/occurrence/search"

PARAMS_BASE = {
    "kingdom":            "Plantae",
    "country":            "BR",
    "hasCoordinate":      "true",
    "hasGeospatialIssue": "false",
    "limit":              300,
}

TOTAL_DESEJADO = 6_000

### 2. Coleta via API REST

Requisições paginadas com tratamento de erros e pausa entre chamadas para respeitar
os limites de uso da API. O loop encerra quando o total desejado é atingido ou
quando não há mais resultados disponíveis.

In [17]:
def coletar_ocorrencias(total_desejado: int) -> list[dict]:
    todos_registros = []
    offset = 0

    while len(todos_registros) < total_desejado:
        params = {**PARAMS_BASE, "offset": offset}

        try:
            response = requests.get(BASE_URL, params=params, timeout=30)
            response.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f"Erro na requisicao (offset={offset}): {e}")
            break

        dados = response.json()
        resultados = dados.get("results", [])

        if not resultados:
            print("Todos os registros disponiveis foram coletados.")
            break

        todos_registros.extend(resultados)

        total_disponivel = dados.get("count", "?")
        print(
            f"Pagina coletada | offset={offset:>6} | "
            f"acumulado: {len(todos_registros):>5} / {total_disponivel} disponiveis"
        )

        offset += 300
        time.sleep(0.5)

    print(f"\nColeta finalizada. Total de registros brutos: {len(todos_registros)}")
    return todos_registros

In [18]:
registros_brutos = coletar_ocorrencias(TOTAL_DESEJADO)

Pagina coletada | offset=     0 | acumulado:   300 / 33358464 disponiveis
Pagina coletada | offset=   300 | acumulado:   600 / 33358464 disponiveis
Pagina coletada | offset=   600 | acumulado:   900 / 33358464 disponiveis
Pagina coletada | offset=   900 | acumulado:  1200 / 33358464 disponiveis
Pagina coletada | offset=  1200 | acumulado:  1500 / 33358464 disponiveis
Pagina coletada | offset=  1500 | acumulado:  1800 / 33358464 disponiveis
Pagina coletada | offset=  1800 | acumulado:  2100 / 33358464 disponiveis
Pagina coletada | offset=  2100 | acumulado:  2400 / 33358464 disponiveis
Pagina coletada | offset=  2400 | acumulado:  2700 / 33358464 disponiveis
Pagina coletada | offset=  2700 | acumulado:  3000 / 33358464 disponiveis
Pagina coletada | offset=  3000 | acumulado:  3300 / 33358464 disponiveis
Pagina coletada | offset=  3300 | acumulado:  3600 / 33358464 disponiveis
Pagina coletada | offset=  3600 | acumulado:  3900 / 33358464 disponiveis
Pagina coletada | offset=  3900 | acum

### 3. Limpeza e Validacao

Etapas aplicadas em sequencia:

- Selecao e renomeacao das colunas relevantes
- Remocao de registros sem coordenadas ou sem especie identificada
- Conversao de tipos numericos (`latitude`, `longitude`, `ano`, `mes`)
- Filtragem pelo bounding box geografico do Brasil
- Normalizacao de texto (strip + title case)
- Deduplicacao por `gbifID`

In [19]:
COLUNAS_RELEVANTES = {
    "species":          "especie",
    "family":           "familia",
    "genus":            "genero",
    "decimalLatitude":  "latitude",
    "decimalLongitude": "longitude",
    "stateProvince":    "estado",
    "municipality":     "municipio",
    "year":             "ano",
    "month":            "mes",
    "basisOfRecord":    "tipo_registro",
    "datasetName":      "dataset",
    "gbifID":           "id_gbif",
}

In [24]:
def limpar_dados(registros_brutos: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(registros_brutos)
    print(f"Shape bruto: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

    # Selecao de colunas
    colunas_existentes = {k: v for k, v in COLUNAS_RELEVANTES.items() if k in df.columns}
    df = df[list(colunas_existentes.keys())].copy()
    df = df.rename(columns=colunas_existentes)

    # Remocao de nulos essenciais
    antes = len(df)
    df = df.dropna(subset=["latitude", "longitude", "especie"])
    print(f"Removidas {antes - len(df):,} linhas sem lat/long/especie")

    # Conversao de tipos
    df["latitude"]  = pd.to_numeric(df["latitude"],  errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df["ano"]       = pd.to_numeric(df["ano"],        errors="coerce").astype("Int64")
    df["mes"]       = pd.to_numeric(df["mes"],        errors="coerce").astype("Int64")

    # Bounding box do Brasil
    antes = len(df)
    df = df[
        (df["latitude"].between(-34.0, 5.5)) &
        (df["longitude"].between(-74.0, -34.0))
    ]
    print(f"Removidas {antes - len(df):,} coordenadas fora do Brasil")

    # Normalizacao de texto
    cols_texto = ["especie", "familia", "genero", "estado", "municipio"]
    for col in cols_texto:
        if col in df.columns:
            df[col] = df[col].str.strip().str.title()

    # Deduplicacao
    if "id_gbif" in df.columns:
        antes = len(df)
        df = df.drop_duplicates(subset=["id_gbif"])
        print(f"Removidas {antes - len(df):,} duplicatas por id_gbif")

    df = df.reset_index(drop=True)

    print(f"Shape final: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
    print(f"Especies unicas: {df['especie'].nunique():,}")
    if "estado" in df.columns:
        print(f"Estados presentes: {df['estado'].nunique()}")

    return df

In [25]:
df = limpar_dados(registros_brutos)
df.head()

Shape bruto: 6,000 linhas x 123 colunas
Removidas 259 linhas sem lat/long/especie
Removidas 7 coordenadas fora do Brasil
Removidas 0 duplicatas por id_gbif
Shape final: 5,734 linhas x 12 colunas
Especies unicas: 2,182
Estados presentes: 28


,especie,familia,genero,latitude,longitude,estado,municipio,ano,mes,tipo_registro,dataset,id_gbif
0,Zeyheria Tuberculosa,Bignoniaceae,Zeyheria,-22.887250,-48.490417,São Paulo,Botucatu,2026,1,PRESERVED_SPECIMEN,NaN,3414152432
1,Mauritia Flexuosa,Arecaceae,Mauritia,-2.704655,-42.872883,Maranhão,NaN,2026,1,HUMAN_OBSERVATION,iNaturalist research-grade observations,5938027332
2,Godartiana Muscosa,Nymphalidae,Godartiana,-20.522878,-41.861437,Minas Gerais,NaN,2026,1,HUMAN_OBSERVATION,iNaturalist research-grade observations,5938027564
3,Sarcoramphus Papa,Cathartidae,Sarcoramphus,-26.306104,-49.155540,Santa Catarina,NaN,2026,1,HUMAN_OBSERVATION,iNaturalist research-grade observations,5938027691
4,Tyrannus Melancholicus,Tyrannidae,Tyrannus,-15.737629,-47.864714,Distrito Federal,NaN,2026,1,HUMAN_OBSERVATION,iNaturalist research-grade observations,5938028534


### 4. Inspecao Rapida do Dataset

Verificacao de tipos, valores nulos e distribuicao inicial antes de salvar.

In [29]:
print(df.dtypes)
print("\n")
print(df.isnull().sum())

especie              str
familia              str
genero               str
latitude         float64
longitude        float64
estado               str
municipio            str
ano                Int64
mes                Int64
tipo_registro        str
dataset              str
id_gbif              str
dtype: object


especie             0
familia             1
genero              0
latitude            0
longitude           0
estado             35
municipio        5724
ano                 0
mes                 0
tipo_registro       0
dataset            40
id_gbif             0
dtype: int64


In [9]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
especie,2856,1333,Sicalis Flaveola,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
familia,2856,397,Thraupidae,146,NaN,NaN,NaN,NaN,NaN,NaN,NaN
genero,2856,1029,Philodendron,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
latitude,2856.0,NaN,NaN,NaN,-20.100069,7.622471,-32.071876,-25.352776,-22.718059,-16.118393,1.040756
longitude,2856.0,NaN,NaN,NaN,-45.775541,5.660591,-72.791834,-49.079088,-46.349346,-41.941726,-34.803649
estado,2824,27,São Paulo,513,NaN,NaN,NaN,NaN,NaN,NaN,NaN
municipio,10,3,Bragança,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ano,2856.0,<NA>,<NA>,<NA>,2026.0,0.0,2026.0,2026.0,2026.0,2026.0,2026.0
mes,2856.0,<NA>,<NA>,<NA>,1.0,0.0,1.0,1.0,1.0,1.0,1.0
tipo_registro,2856,2,HUMAN_OBSERVATION,2846,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 5. Armazenamento

Os dados sao salvos em dois formatos:

- `.parquet` - por ser colunar e preservar a tipagem
- `.csv` - por ser legivel por qualquer ferramenta 

In [ ]:
def salvar_dados(df: pd.DataFrame, pasta: str = "dados") -> None:
    os.makedirs(pasta, exist_ok=True)

    caminho_parquet = os.path.join(pasta, "flores_brasil.parquet")
    caminho_csv     = os.path.join(pasta, "flores_brasil.csv")

    df.to_parquet(caminho_parquet, index=False)
    df.to_csv(caminho_csv, index=False, encoding="utf-8-sig")

    kb_parquet = os.path.getsize(caminho_parquet) / 1024
    kb_csv     = os.path.getsize(caminho_csv) / 1024

In [11]:
salvar_dados(df, pasta="dados")

Dados salvos em 'dados/'
  flores_brasil.parquet     119.9 KB
  flores_brasil.csv         423.6 KB
  Reducao: parquet e 3.5x menor que o CSV


### 6. Resumo

Visao geral do dataset coletado

In [32]:
print(f"\nRegistros totais :  {len(df):,}")
print(f"Especies unicas  :  {df['especie'].nunique():,}")

if "familia" in df.columns:
    print(f"Familias unicas  :  {df['familia'].nunique():,}")

if "estado" in df.columns:
    print(f"Estados com dados:  {df['estado'].nunique()}")

if "ano" in df.columns:
    print(f"\nPeriodo dos dados:")
    print(f"  Ano mais antigo : {df['ano'].min()}")
    print(f"  Ano mais recente: {df['ano'].max()}")

print(f"\nTop 5 especies mais registradas:")
for especie, count in df["especie"].value_counts().head(5).items():
    print(f"  {especie:<42} {count:>5} registros")

if "estado" in df.columns:
    print(f"\nTop 5 estados com mais registros:")
    for estado, count in df["estado"].value_counts().head(5).items():
        print(f"  {estado:<30} {count:>5} registros")


Registros totais :  5,734
Especies unicas  :  2,182
Familias unicas  :  522
Estados com dados:  28

Periodo dos dados:
  Ano mais antigo : 2026
  Ano mais recente: 2026

Top 5 especies mais registradas:
  Sicalis Flaveola                              58 registros
  Pitangus Sulphuratus                          41 registros
  Trichonephila Clavipes                        40 registros
  Nycticorax Nycticorax                         38 registros
  Columbina Talpacoti                           36 registros

Top 5 estados com mais registros:
  São Paulo                       1154 registros
  Rio De Janeiro                   624 registros
  Rio Grande Do Sul                563 registros
  Minas Gerais                     490 registros
  Santa Catarina                   458 registros
